# Multiple Linear Regression: Revenue Prediction
## Clean the raw data → 80% train / 20% test → fit → evaluate

**Objective:** predict Revenue using Ad_spend, Visitors, and Discount. ID identifies rows and is excluded from the model inputs.

This worked notebook uses your supplied data. After cleaning, 10 records remain; 8 are used for training and 2 for testing with `random_state=42`. Ordinary least squares estimates the coefficients using training data only.

**Assumptions:** Discount is a percentage between 0 and 100; advertising spend and visitor counts cannot be negative. Currency and reporting periods were not supplied, so monetary values are described in dataset units.

**Run:** select a Python kernel in VS Code or Jupyter and run the cells in order. If imports fail, uncomment and run the installation cell, then restart the kernel. Outputs are generated when you run the notebook.

In [ ]:
# Uncomment only if the packages are missing from your notebook environment.
# %pip install numpy pandas matplotlib scikit-learn

In [ ]:
from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.precision", 4)
FEATURES = ["Ad_spend", "Visitors", "Discount"]
TARGET = "Revenue"
RANDOM_STATE = 42

## 1. Load the 14 raw records
Keep the original data available for comparison and record why each excluded row is removed.

In [ ]:
RAW_CSV = """ID,Ad_spend,Visitors,Discount,Revenue
M01,10,100,5,67
M02,10,100,15,57
M03,10,200,5,93
M04,10,200,15,83
M05,20,100,5,83
M06,20,100,15,73
M07,20,200,5,117
M08,20,200,15,107
M09,12,140,8,79
M10,18,180,12,97
M03,10,200,5,93
M11,,150,10,80
M12,-5,130,10,60
M13,15,160,120,75
"""
raw = pd.read_csv(StringIO(RAW_CSV))
print("Raw shape:", raw.shape)
display(raw)
display(raw.dtypes.rename("Data type").to_frame())
display(raw.isna().sum().rename("Missing values").to_frame())

## 2. Clean with explicit rules

| Record | Issue | Action |
|---|---|---|
| Second M03 | Exact duplicate | Keep the first occurrence |
| M11 | Missing advertising spend | Remove for this exercise |
| M12 | Negative advertising spend | Remove |
| M13 | Discount outside 0–100% | Remove |

These rules are applied before splitting. They do not estimate statistics from the data. We do not remove a valid row simply because it has an unusual Revenue. Dropping missing records is an exercise choice; in larger datasets, assess the impact and consider training-only imputation.

In [ ]:
numeric_columns = FEATURES + [TARGET]
data = raw.copy()
data[numeric_columns] = data[numeric_columns].apply(pd.to_numeric, errors="coerce")

duplicate = data.duplicated(keep="first")
missing = data[numeric_columns].isna().any(axis=1) | data["ID"].isna()
negative_spend = data["Ad_spend"].lt(0)
negative_visitors = data["Visitors"].lt(0)
invalid_discount = data["Discount"].notna() & ~data["Discount"].between(0, 100)

reasons = pd.Series("", index=data.index, dtype="object")
for mask, reason in [
    (duplicate, "Exact duplicate"),
    (missing, "Missing/non-numeric required value"),
    (negative_spend, "Negative Ad_spend"),
    (negative_visitors, "Negative Visitors"),
    (invalid_discount, "Discount outside 0-100%"),
]:
    reasons.loc[mask] = reasons.loc[mask] + reason + "; "

removed = data.loc[reasons.ne("")].copy()
removed.insert(0, "CSV_data_row", removed.index + 1)
removed["Reason"] = reasons.loc[removed.index].str.rstrip("; ")
clean = data.loc[reasons.eq("")].reset_index(drop=True)

print("Removed records:")
display(removed)
print(f"Kept {len(clean)} of {len(raw)} records:")
display(clean)

assert len(raw) == 14 and len(removed) == 4 and len(clean) == 10
assert clean["ID"].is_unique
assert not clean[numeric_columns].isna().any().any()
assert clean["ID"].tolist() == [f"M{i:02d}" for i in range(1, 11)]

## 3. Split: 8 training records and 2 test records

Split whole rows so IDs remain aligned with inputs and targets. A fixed seed makes the split repeatable. Fit the model using the training set; reserve the test set for final evaluation. No stratification is needed for this continuous target.

In [ ]:
train, test = train_test_split(
    clean, test_size=0.20, random_state=RANDOM_STATE, shuffle=True
)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"Training: {len(train)} records ({len(train) / len(clean):.0%})")
display(train)
print(f"Testing: {len(test)} records ({len(test) / len(clean):.0%})")
display(test)
assert len(train) == 8 and len(test) == 2
assert set(train["ID"]).isdisjoint(set(test["ID"]))

## 4. Explore only the training data

Inspect the ranges and pairwise plots. Each scatter plot shows a marginal relationship, while a multiple-regression coefficient describes a relationship holding the other predictors fixed.

In [ ]:
display(train[FEATURES + [TARGET]].describe())
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, feature in zip(axes, FEATURES):
    ax.scatter(train[feature], y_train, color="steelblue", s=55)
    ax.set(xlabel=feature, ylabel="Revenue", title=f"Revenue vs {feature}")
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Fit multiple linear regression

The model is:

\[
\widehat{Revenue} = b_0 + b_1 Ad\_spend + b_2 Visitors + b_3 Discount
\]

Ordinary least squares chooses the coefficients that minimize the **sum of squared training residuals**. A residual is actual Revenue minus predicted Revenue.

Scaling is not required for this unregularized least-squares fit. The model uses the original units so that its coefficients are easy to interpret. This is an additive model; it does not include interactions or polynomial terms.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

coefficients = pd.DataFrame({
    "Term": ["Intercept"] + FEATURES,
    "Coefficient": [model.intercept_, *model.coef_],
})
display(coefficients)
equation = f"Predicted Revenue = {model.intercept_:.6f}"
for feature, value in zip(FEATURES, model.coef_):
    equation += f" {value:+.6f} * {feature}"
print(equation)

## 6. Understand and verify the coefficient calculation

For a design matrix A containing an initial column of ones and the three predictor columns, least squares solves A × b ≈ y. The column of ones represents the intercept.

The next cell independently calculates the same coefficients with NumPy's least-squares solver. This avoids explicitly inverting AᵀA and checks that the design matrix has enough independent columns.

In [ ]:
A_train = np.column_stack([np.ones(len(X_train)), X_train.to_numpy()])
beta, residual_sums, rank, singular_values = np.linalg.lstsq(
    A_train, y_train.to_numpy(), rcond=None
)
display(pd.DataFrame(A_train, columns=["Intercept column"] + FEATURES,
                     index=train["ID"]))
print("Design matrix shape:", A_train.shape)
print("Design matrix rank:", rank)
print("NumPy coefficients:", beta)
assert rank == A_train.shape[1], "The design matrix does not have full column rank."
np.testing.assert_allclose(
    beta, np.r_[model.intercept_, model.coef_], rtol=1e-8, atol=1e-8
)
print("NumPy and scikit-learn coefficients agree.")

In [ ]:
print(f"Intercept: {model.intercept_:.4f} Revenue units at all-zero inputs.")
print("All-zero inputs are outside the observed data; interpret the intercept cautiously.")
for feature, coefficient in zip(FEATURES, model.coef_):
    unit = "percentage point" if feature == "Discount" else "unit"
    print(
        f"{feature}: a 1-{unit} increase changes predicted Revenue "
        f"by {coefficient:+.4f} units, holding the other inputs fixed."
    )
print("These are fitted associations, not evidence of causal effects.")

## 7. Predict the two held-out records

The test records have not contributed to the fitted coefficients. The table preserves their IDs, predictors, actual values, predictions, and errors.

In [ ]:
test_prediction = model.predict(X_test)
results = test[["ID"] + FEATURES].copy()
results["Actual_Revenue"] = y_test
results["Predicted_Revenue"] = test_prediction
results["Residual"] = results["Actual_Revenue"] - results["Predicted_Revenue"]
results["Absolute_Error"] = results["Residual"].abs()
results["Squared_Error"] = results["Residual"] ** 2
display(results.reset_index(drop=True))

first = X_test.iloc[0]
contributions = pd.Series(model.coef_, index=FEATURES) * first
print("Worked prediction for", test.iloc[0]["ID"])
print(f"Intercept contribution: {model.intercept_:.6f}")
display(contributions.rename("Coefficient x input").to_frame())
print(f"Total prediction: {model.intercept_ + contributions.sum():.6f}")
print(f"Actual Revenue: {y_test.iloc[0]:.6f}")

## 8. Evaluate test performance

- **MAE:** average absolute error, in Revenue units.
- **RMSE:** square root of average squared error, also in Revenue units; larger errors carry greater weight.
- **R²:** 1 − (sum of squared residuals / sum of squared deviations from the test mean). It can be negative.

We also compare against a baseline that predicts the **training mean** for both test records. The test mean in the R² definition is a scoring reference, not a fitted predictor.

**Only two test observations are available.** These scores describe this split and are not a reliable estimate of performance on a large unseen population. Training metrics are shown separately and are not substitutes for test performance.

In [ ]:
def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R_squared": r2_score(actual, predicted),
    }

train_prediction = model.predict(X_train)
baseline_prediction = np.full(len(y_test), y_train.mean())
metrics = pd.DataFrame({
    "Training - fitted model": regression_metrics(y_train, train_prediction),
    "Test - fitted model": regression_metrics(y_test, test_prediction),
    "Test - training-mean baseline": regression_metrics(y_test, baseline_prediction),
}).T
display(metrics)

errors = y_test.to_numpy() - test_prediction
mae_manual = np.abs(errors).mean()
rmse_manual = np.sqrt(np.mean(errors ** 2))
sse = np.sum(errors ** 2)
sst = np.sum((y_test.to_numpy() - y_test.mean()) ** 2)
r2_manual = 1 - sse / sst
print(f"Test MAE = {mae_manual:.6f}")
print(f"Test RMSE = {rmse_manual:.6f}")
print(f"Test R² = 1 - {sse:.6f} / {sst:.6f} = {r2_manual:.6f}")
np.testing.assert_allclose(
    [mae_manual, rmse_manual, r2_manual],
    list(regression_metrics(y_test, test_prediction).values())
)

## 9. Visualize predictions and training residuals

With three predictors there is no single 2D regression line that represents the full model. Instead, compare actual and predicted Revenue, and inspect residuals against fitted values. Eight training residuals are too few for strong diagnostic conclusions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_train, train_prediction, label="Training", alpha=0.7)
axes[0].scatter(y_test, test_prediction, label="Test", marker="s", s=85)
for row in results.itertuples():
    axes[0].annotate(row.ID, (row.Actual_Revenue, row.Predicted_Revenue),
                     xytext=(5, 5), textcoords="offset points")
low = min(y_train.min(), y_test.min(), train_prediction.min(), test_prediction.min()) - 3
high = max(y_train.max(), y_test.max(), train_prediction.max(), test_prediction.max()) + 3
axes[0].plot([low, high], [low, high], "k--", label="Perfect prediction")
axes[0].set(xlabel="Actual Revenue", ylabel="Predicted Revenue",
            title="Actual vs predicted")
axes[0].legend()
axes[1].scatter(train_prediction, y_train.to_numpy() - train_prediction)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(xlabel="Fitted training Revenue", ylabel="Residual (actual - predicted)",
            title="Training residuals")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 10. Predict a new example

Use the same feature names and order. This example lies inside the individual training ranges; that alone does not guarantee an accurate prediction. The actual Revenue is unknown, so it is not part of the evaluation.

In [ ]:
new_example = pd.DataFrame([{
    "Ad_spend": 15,
    "Visitors": 150,
    "Discount": 10,
}])[FEATURES]
new_result = new_example.copy()
new_result["Predicted_Revenue"] = model.predict(new_example)
display(new_result)

## Conclusion

The workflow removes one duplicate and three invalid/incomplete records, then fits an additive model on eight observations and evaluates it on two held-out observations. Use the printed equation and test table as the numerical solution.

A high score on this tiny test set does not establish a strong model for real business use. Collect more observations and evaluate across representative conditions before making broader claims. Do not repeatedly choose model settings based on these two test rows.

The notebook keeps the supplied data inline and does not download a dataset or modify the existing gradient-descent notebook.